In [5]:
!pip install --upgrade transformers
!pip install torch
!pip install torchvision
!pip install pillow


In [6]:
from pathlib import Path

image_dir = Path("my_images")
pics = sorted(
    str(p)
    for p in image_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
)

model_names = [
    "Salesforce/blip-image-captioning-large",
    "Salesforce/blip-vqa-base",
    "Salesforce/blip-image-captioning-base"
 ]

In [7]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import requests
from io import BytesIO
from pathlib import Path

loaded_models = {}

for m_name in model_names:
    print(f"Loading {m_name}...")
    loaded_models[m_name] = {
        "processor": BlipProcessor.from_pretrained(m_name),
        "model": BlipForConditionalGeneration.from_pretrained(m_name)
    }

def _load_image(image_ref: str) -> Image.Image:
    if image_ref.startswith("http://") or image_ref.startswith("https://"):
        response = requests.get(image_ref, timeout=30)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert("RGB")
    image_path = Path(image_ref)
    return Image.open(image_path).convert("RGB")

def generate_image_description(model_obj, image_ref: str):
    try:
        img = _load_image(image_ref)
        inputs = model_obj["processor"](images=img, return_tensors="pt")
        out = model_obj["model"].generate(**inputs)
        return model_obj["processor"].decode(out[0], skip_special_tokens=True)
    except Exception as e:
        return str(e)


Loading Salesforce/blip-image-captioning-large...


Loading weights: 100%|██████████| 616/616 [00:00<00:00, 5225.47it/s]


Loading Salesforce/blip-vqa-base...


Loading weights: 100%|██████████| 472/472 [00:00<00:00, 5574.59it/s]
[transformers] BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-vqa-base
Key                                                                        | Status     |  | 
---------------------------------------------------------------------------+------------+--+-
text_encoder.encoder.layer.{0...11}.attention.output.LayerNorm.weight      | UNEXPECTED |  | 
text_encoder.encoder.layer.{0...11}.output.LayerNorm.weight                | UNEXPECTED |  | 
text_encoder.encoder.layer.{0...11}.intermediate.dense.bias                | UNEXPECTED |  | 
text_encoder.encoder.layer.{0...11}.attention.self.value.weight            | UNEXPECTED |  | 
text_encoder.encoder.layer.{0...11}.attention.output.LayerNorm.bias        | UNEXPECTED |  | 
text_encoder.encoder.layer.{0...11}.intermediate.dense.weight              | UNEXPECTED |  | 
text_encoder.encoder.layer.{0...11}.crossattention.output.LayerNorm.weight | UNEXPECTED |  |

Loading Salesforce/blip-image-captioning-base...


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 29528.12it/s]


In [8]:
import pandas as pd
from IPython.display import HTML

# 1. Collect your data
data = []
for url in pics:
    row = {"Image": url}
    for model_name, model_obj in loaded_models.items():
        description = generate_image_description(model_obj, url)
        short_name = model_name
        row[short_name] = description
    data.append(row)

# 2. Create the DataFrame
df = pd.DataFrame(data)

# 3. Helper function to turn a URL into an HTML image tag
def path_to_image_html(path):
    return f'<img src="{path}" width="150" >'

# 4. Render the table with images
HTML(df.to_html(escape=False, formatters=dict(Image=path_to_image_html)))

/home/user/LLMs/.venv/lib/python3.13/site-packages/transformers/generation/utils.py:1590: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


,Image,Salesforce/blip-image-captioning-large,Salesforce/blip-vqa-base,Salesforce/blip-image-captioning-base
0,,there are two birds sitting on a tree branch in the snow,black bird,a couple of birds are sitting in a tree
1,,there is a duck sitting on a branch in the middle of the forest,bird,a bird perched on a branch of a tree
2,,a close up of a computer board with many electronic components,electronic board,a computer board with many components on it
3,,there is a bird that is sitting on a wire in the sky,bird,a bird is perched on a wire in the middle of a forest
4,,there are many birds sitting in the branches of a tree,birds,a bear is climbing in a tree
5,,people standing around a table with a computer and a monitor,table,a group of people standing around a table with a laptop
6,,there are two ducks that are standing on the ground,bird,two ducks are standing on the ground
7,,there are many padlocks on the railing of a balcony overlooking a river,railing,a bench overlooking a river and a forest
8,,sunset over a train station with a person walking on the platform,sun is setting,a train station with a train on the tracks
9,,there is a squirrel that is sitting on a tree in the woods,squirrel,a squirrel is climbing up a tree


Geriausias modelis:
- Salesforce/blip-image-captioning-large

Pasidomėjus kodėl, pasidaro akivaizdu:
- Turi 470 mln. parametrų, lyginant su 224 mln. parametrų modeliais ("Salesforce/blip-vqa-base", "Salesforce/blip-image-captioning-base").
- Naudoja daugiau RAM, yra letesnis, bet galingesnis.

Dar geresnė, galingesnė alternatyva būtų BLIP 2, BLIP 3, tačiau jos reikalautų brangesnio GPU, daugiau RAM.